# OSL End-to-End Notebook (Connectome)

This notebook is dedicated to one setup only:
- `Odor2DEnvConnectome`
- Connectome actor + recurrent critic
- fixed reward shaping

Only the main training knobs are exposed below. Everything else is fixed inside the notebook.


In [ ]:
import io
import json
import os
import random
import time
from pathlib import Path
from types import SimpleNamespace

import gymnasium as gym
import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium import spaces
from matplotlib import cm
from torch.distributions import Bernoulli
from torch.distributions import Normal

try:
    from IPython.display import Image as DisplayImage, display
except Exception:
    DisplayImage = None
    display = None

try:
    from PIL import Image, ImageDraw
except Exception:
    Image = None
    ImageDraw = None

print("Notebook-local imports ready")


In [ ]:
# Main knobs + shared helpers.
OUT_DIR = "runs"
RUN_NAME = None
FORCE_CPU = False

SEED = 42
TOTAL_EPISODES = 20
EVAL_EPISODES = 100
CRITIC_HIDDEN = 147
CONNECTOME_HIDDEN = 180
CONNECTOME_STEPS = 4
BATCH_SIZE = 128
SEQ_LEN = 16
LEARNING_STARTS = 5000
LR_ACTOR = 3e-4
LR_CRITIC = 3e-4
LR_ALPHA = 3e-4
LOG_EVERY = 20

# Fixed notebook defaults.
BUFFER_SIZE = 150000
GAMMA = 0.99
TAU = 0.005
EVAL_SEED_BASE = 20000

LAST_RUN_DIR = None
LAST_EVAL_RESULT = None

ENV_CONFIG = {
    "L": 3.0,
    "dt": 0.1,
    "src_x": 0.0,
    "src_y": 0.0,
    "wind_x": 0.0,
    "wind_y": 0.0,
    "sigma_c": 1.0,
    "r_goal": 0.35,
}


def make_args(run_name=None, out_dir=None):
    if out_dir is None:
        out_dir = OUT_DIR
    if run_name is None:
        run_name = RUN_NAME
    if run_name is None:
        run_name = time.strftime("connectome_notebook_%Y%m%d_%H%M%S")

    return SimpleNamespace(
        out_dir=out_dir,
        run_name=run_name,
        seed=SEED,
        total_episodes=TOTAL_EPISODES,
        eval_episodes=EVAL_EPISODES,
        critic_hidden=CRITIC_HIDDEN,
        connectome_hidden=CONNECTOME_HIDDEN,
        connectome_steps=CONNECTOME_STEPS,
        batch_size=BATCH_SIZE,
        seq_len=SEQ_LEN,
        learning_starts=LEARNING_STARTS,
        lr_actor=LR_ACTOR,
        lr_critic=LR_CRITIC,
        lr_alpha=LR_ALPHA,
        log_every=LOG_EVERY,
    )


def set_seed(seed, deterministic_torch=True):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic_torch:
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


class ReplayBuffer:
    def __init__(self, cap_steps=150000):
        self.cap_steps = int(cap_steps)
        self.episodes = []
        self.n_steps = 0

    def add_episode(self, ep):
        if not ep:
            return
        self.episodes.append(ep)
        self.n_steps += len(ep)
        while self.n_steps > self.cap_steps and self.episodes:
            removed = self.episodes.pop(0)
            self.n_steps -= len(removed)

    def __len__(self):
        return self.n_steps

    def sample_continuous(self, batch_size, seq_len):
        batch_size = int(batch_size)
        seq_len = int(seq_len)
        candidates = [ep for ep in self.episodes if len(ep) >= seq_len + 1]
        if not candidates:
            raise ValueError("Not enough data to sample continuous sequences.")

        obs_batch, act_batch, rew_batch, terminal_batch = [], [], [], []
        for _ in range(batch_size):
            ep = random.choice(candidates)
            s = random.randint(0, len(ep) - seq_len - 1)
            chunk = ep[s : s + seq_len]
            next_chunk = ep[s + 1 : s + seq_len + 1]

            obs_seq = [step[0] for step in chunk] + [next_chunk[-1][3]]
            act_seq = [step[1] for step in chunk]
            rew_seq = [step[2] for step in chunk]
            terminal_seq = [step[4] for step in chunk]

            obs_batch.append(np.asarray(obs_seq, dtype=np.float32))
            act_batch.append(np.asarray(act_seq, dtype=np.float32))
            rew_batch.append(np.asarray(rew_seq, dtype=np.float32))
            terminal_batch.append(np.asarray(terminal_seq, dtype=np.float32))

        return (
            np.stack(obs_batch, axis=0),
            np.stack(act_batch, axis=0),
            np.stack(rew_batch, axis=0),
            np.stack(terminal_batch, axis=0),
        )


def current_env_config():
    return dict(ENV_CONFIG)


def run_dir_from_args(args):
    return os.path.abspath(os.path.join(args.out_dir, args.run_name))


def build_run_config(args):
    return {
        "run_name": args.run_name,
        "seed": int(args.seed),
        "total_episodes": int(args.total_episodes),
        "eval_episodes": int(args.eval_episodes),
        "critic_hidden": int(args.critic_hidden),
        "connectome_hidden": int(args.connectome_hidden),
        "connectome_steps": int(args.connectome_steps),
        "batch_size": int(args.batch_size),
        "seq_len": int(args.seq_len),
        "learning_starts": int(args.learning_starts),
        "lr_actor": float(args.lr_actor),
        "lr_critic": float(args.lr_critic),
        "lr_alpha": float(args.lr_alpha),
        "log_every": int(args.log_every),
        "env": current_env_config(),
    }


def write_run_config(run_dir, args):
    cfg = build_run_config(args)
    config_json_path = Path(run_dir) / "config.json"
    config_json_path.write_text(json.dumps(cfg, indent=2))
    return cfg


def load_run_config(run_dir):
    config_path = Path(run_dir) / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"Missing config.json in {run_dir}")
    return json.loads(config_path.read_text())


def resolve_run_dir(run_dir=None):
    if run_dir:
        return os.path.abspath(run_dir)
    if globals().get("run_dir"):
        return os.path.abspath(globals()["run_dir"])
    if globals().get("LAST_RUN_DIR"):
        return os.path.abspath(globals()["LAST_RUN_DIR"])
    if globals().get("args") is not None:
        return run_dir_from_args(globals()["args"])
    return None


preview_args = make_args()
print("RUN_NAME:", preview_args.run_name)
print("Fixed setup: Odor2DEnvConnectome / Connectome2")
print("Default episodes:", preview_args.total_episodes, "| Eval episodes:", preview_args.eval_episodes)


In [ ]:
# Ported: src/envs/odor_env_v4.py



class Odor2DEnvConnectome(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(
        self,
        render_mode=None,
        L=3.0,
        dt=0.1,
        src_x=0.0,
        src_y=0.0,
        wind_x=0.0,
        wind_y=0.0,
        sigma_c=1.0,
        r_goal=0.35,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.L = float(L)
        self.dt = float(dt)
        self.ds = 0.08
        self.src_x = float(np.clip(float(src_x), -self.L, self.L))
        self.src_y = float(np.clip(float(src_y), -self.L, self.L))
        self.wind_x = float(wind_x)
        self.wind_y = float(wind_y)
        self._wind_mag = float(np.hypot(self.wind_x, self.wind_y))

        if self._wind_mag > 1e-6:
            self._wind_dir = (self.wind_x / self._wind_mag, self.wind_y / self._wind_mag)
        else:
            self._wind_dir = (0.0, 0.0)

        self.sigma_c = float(sigma_c)
        self.r_goal = float(r_goal)
        self.b_hold = 0.5
        self.b_oob = 5.0
        self.max_steps = 300
        self.bg_c = 0.0
        self.sensor_noise = 0.01

        self.v_min = 0.0
        self.v_max = 0.45
        self.omega_max = 5.0
        self.accel_max = 1.2
        self.omega_accel_max = 50.0
        self.control_penalty = 0.01
        self.turn_penalty = 0.01
        self.cast_penalty = 0.025
        self.reward_scale = 0.5
        self.goal_hold_steps = 20
        self.terminate_on_hold = True
        self.turn_requires_cast = True
        self.turn_window_steps = 1

        self.np_random = np.random.default_rng(None)

        # action = [v_cmd, omega_cmd, cast_cmd]
        self.action_space = spaces.Box(
            low=np.array([self.v_min, -self.omega_max, 0.0], dtype=np.float32),
            high=np.array([self.v_max, self.omega_max, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

        # Per-step observation: [concentration, mode]
        # mode: 0=run, 1=cast
        self.obs_step_dim = 2
        obs_low = np.array([0.0, 0.0], dtype=np.float32)
        obs_high = np.array([1.0, 1.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float32)

        self._step = 0
        self.x = 0.0
        self.y = 0.0
        self.th = 0.0
        self.v = 0.0
        self.omega = 0.0

        self.in_cast = False
        self.cast_phase = 0
        self.turn_steps_left = 0
        self._scan_dirs = np.array([np.pi / 2, -np.pi / 2], dtype=np.float32)
        self._scan_seq = (0, 1, 0, 1)
        self._scan_c = np.zeros(4, dtype=np.float32)
        self.goal_hold_count = 0
        self.prev_in_goal = False

        self._sense_pt = None
        self._trail = []
        self._img_size = 360
        self._heatmap_img = None
        self._cbar_img = None
        self._last_obs = np.zeros((self.obs_step_dim,), dtype=np.float32)

    def _conc(self, x, y):
        x = np.asarray(x, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        xr = x - np.float32(self.src_x)
        yr = y - np.float32(self.src_y)

        if self._wind_mag <= 1e-6:
            r2 = xr * xr + yr * yr
            c = np.exp(-r2 / (2.0 * self.sigma_c * self.sigma_c))
        else:
            wx, wy = self._wind_dir
            t = xr * wx + yr * wy
            s = -xr * wy + yr * wx
            stretch = 1.0 + min(self._wind_mag, 2.0)
            sigma_s = self.sigma_c
            sigma_t = self.sigma_c * stretch
            sigma_up = self.sigma_c / stretch
            t_pos = np.maximum(0.0, t)
            t_neg = np.maximum(0.0, -t)
            c = np.exp(-((s * s) / (2.0 * sigma_s ** 2) + (t_pos ** 2) / (2.0 * sigma_t ** 2) + (t_neg ** 2) / (2.0 * sigma_up ** 2)))

        c = self.bg_c + (1.0 - self.bg_c) * c
        return np.clip(c, 0.0, 1.0)

    def _sense(self, phi=0.0):
        ang = float(self.th + phi)
        sx = float(self.x + np.cos(ang) * self.ds)
        sy = float(self.y + np.sin(ang) * self.ds)
        c = self._conc(sx, sy)
        if self.sensor_noise > 0:
            c += float(self.np_random.normal(0, self.sensor_noise))
        self._sense_pt = (sx, sy)
        return float(np.clip(c, 0.0, 1.0))

    def _set_obs(self, c, mode):
        self._last_obs = np.array([float(c), float(mode)], dtype=np.float32)

    def _get_obs(self):
        return self._last_obs.copy()

    def _norm_angle(self, a):
        return (a + np.pi) % (2 * np.pi) - np.pi

    def _cast_step(self):
        i = int(self.cast_phase)
        side = int(self._scan_seq[i])
        phi = float(self._scan_dirs[side])
        c = self._sense(phi)
        self._scan_c[i] = float(c)

        self.cast_phase += 1
        if self.cast_phase >= 4:
            self.in_cast = False
            self.cast_phase = 0
            self._scan_c[:] = 0.0
            self.turn_steps_left = self.turn_window_steps
        return c

    def reset(self, seed=None, options=None):
        del options
        if seed is not None:
            self.np_random = np.random.default_rng(seed)

        spawn_margin = 0.25
        spawn_radius_tries = 80
        spawn_angle_tries = 80
        r_min, r_max = max(self.r_goal + spawn_margin, 0.6), min(0.8 * self.L, self.L - spawn_margin)
        cx, cy = self.src_x, self.src_y

        x0, y0 = float(cx), float(cy)
        found = False
        for _ in range(max(1, int(spawn_radius_tries))):
            r0 = float(self.np_random.uniform(r_min, r_max))
            for _ in range(max(1, int(spawn_angle_tries))):
                ang = float(self.np_random.uniform(-np.pi, np.pi))
                tx = float(cx + r0 * np.cos(ang))
                ty = float(cy + r0 * np.sin(ang))
                if (-self.L + spawn_margin) <= tx <= (self.L - spawn_margin) and (-self.L + spawn_margin) <= ty <= (self.L - spawn_margin):
                    x0, y0 = tx, ty
                    found = True
                    break
            if found:
                break
        if not found:
            x0 = float(np.clip(cx, -self.L + spawn_margin, self.L - spawn_margin))
            y0 = float(np.clip(cy, -self.L + spawn_margin, self.L - spawn_margin))

        self.x, self.y = float(x0), float(y0)
        self.th = self.np_random.uniform(-np.pi, np.pi)
        self.v = 0.0
        self.omega = 0.0
        self._step = 0
        self.in_cast = False
        self.cast_phase = 0
        self.turn_steps_left = 0
        self._scan_c[:] = 0.0
        self.goal_hold_count = 0
        self.prev_in_goal = False
        self._trail = [(self.x, self.y)]

        c0 = self._sense(0.0)
        self._set_obs(c0, mode=0.0)
        return self._get_obs(), {}

    def step(self, action):
        self._step += 1
        prev_c = float(self._last_obs[0])
        prev_in_goal = bool(self.prev_in_goal)
        a = np.asarray(action, dtype=np.float32)
        a = np.clip(a, self.action_space.low, self.action_space.high)
        v_cmd = float(a[0])
        omega_cmd = float(a[1])
        cast_cmd = int(np.rint(np.clip(a[2], 0.0, 1.0)))

        did_cast = False
        cast_started = False
        can_turn_now = (not self.turn_requires_cast) or (self.turn_steps_left > 0)

        if self.in_cast:
            c = self._cast_step()
            self.v = 0.0
            self.omega = 0.0
            did_cast = True
        elif cast_cmd == 1:
            self.in_cast = True
            self.cast_phase = 0
            self._scan_c[:] = 0.0
            cast_started = True
            c = self._cast_step()
            self.v = 0.0
            self.omega = 0.0
            did_cast = True
        else:
            if not can_turn_now:
                omega_cmd = 0.0

            dv = np.clip(v_cmd - self.v, -self.accel_max * self.dt, self.accel_max * self.dt)
            domega = np.clip(omega_cmd - self.omega, -self.omega_accel_max * self.dt, self.omega_accel_max * self.dt)

            self.v = float(np.clip(self.v + dv, self.v_min, self.v_max))
            self.omega = float(np.clip(self.omega + domega, -self.omega_max, self.omega_max))

            self.th = self._norm_angle(self.th + self.omega * self.dt)
            self.x += self.v * np.cos(self.th) * self.dt
            self.y += self.v * np.sin(self.th) * self.dt

            c = self._sense(0.0)

            if self.turn_requires_cast and self.turn_steps_left > 0:
                self.turn_steps_left -= 1

        delta_c = float(c - prev_c)
        obs_mode = 1.0 if did_cast else 0.0
        self._set_obs(c, mode=obs_mode)

        oob = (abs(self.x) > self.L) or (abs(self.y) > self.L)
        terminated = bool(oob)
        truncated = bool(self._step >= self.max_steps)

        dx, dy = self.x - self.src_x, self.y - self.src_y
        d = np.hypot(dx, dy)
        in_goal = bool(d < self.r_goal)
        if in_goal:
            self.goal_hold_count += 1
        else:
            self.goal_hold_count = 0
        goal_exited = bool((not in_goal) and prev_in_goal)
        success_hold = bool(self.goal_hold_count >= self.goal_hold_steps)
        self.prev_in_goal = in_goal

        r = float(self.reward_scale * c)
        if did_cast:
            r -= self.cast_penalty
        else:
            r -= self.control_penalty * (abs(self.v) / (self.v_max + 1e-6))
            r -= self.turn_penalty * (abs(self.omega) / (self.omega_max + 1e-6))
        if in_goal:
            r += self.b_hold
        if oob:
            r -= self.b_oob
        if self.terminate_on_hold and success_hold:
            terminated = True

        info = {
            "d": d,
            "c": float(c),
            "in_goal": int(in_goal),
            "goal_hold_count": int(self.goal_hold_count),
            "goal_exited": int(goal_exited),
            "success_hold": int(success_hold),
            "delta_c": delta_c,
            "src_x": self.src_x,
            "src_y": self.src_y,
            "v": self.v,
            "omega": self.omega,
            "did_cast": int(did_cast),
            "cast_start": int(cast_started),
            "cast_cmd": cast_cmd,
            "in_cast": int(self.in_cast),
            "can_turn": int(can_turn_now),
            "turn_steps_left": int(self.turn_steps_left),
        }
        self._trail.append((self.x, self.y))
        return self._get_obs(), float(r), terminated, truncated, info

    def _world_to_px(self, x, y):
        size = self._img_size - 1
        px = int(np.clip((float(x) + self.L) / (2.0 * self.L) * size, 0, size))
        py = int(np.clip((self.L - float(y)) / (2.0 * self.L) * size, 0, size))
        return px, py

    def _build_heatmap(self):
        size = self._img_size
        xs = np.linspace(-self.L, self.L, size, dtype=np.float32)
        ys = np.linspace(self.L, -self.L, size, dtype=np.float32)
        X, Y = np.meshgrid(xs, ys)
        c = self._conc(X, Y).astype(np.float32)
        try:
            rgb = cm.inferno(c)[..., :3].astype(np.float32)
            return np.clip(rgb * 255.0, 0.0, 255.0).astype(np.uint8)
        except Exception:
            r = np.clip(4.0 * c - 1.5, 0.0, 1.0)
            g = np.clip(4.0 * c - 2.5, 0.0, 1.0)
            b = np.clip(4.0 * c - 3.5, 0.0, 1.0)
            return (np.stack([r, g, b], axis=-1) * 255.0).astype(np.uint8)

    def render(self):
        if self.render_mode not in (None, "rgb_array"):
            return None
        if Image is None or ImageDraw is None:
            if self._heatmap_img is None:
                self._heatmap_img = self._build_heatmap()
            return self._heatmap_img.copy()

        W = H = self._img_size
        if self._heatmap_img is None or self._heatmap_img.shape[:2] != (H, W):
            self._heatmap_img = self._build_heatmap()

        img = Image.fromarray(self._heatmap_img.copy(), mode="RGB")
        draw = ImageDraw.Draw(img)

        draw.rectangle((0, 0, W - 1, H - 1), outline=(255, 255, 255), width=1)

        src_px, src_py = self._world_to_px(self.src_x, self.src_y)
        rg = max(1, int(round(self.r_goal / (2.0 * self.L) * (W - 1))))
        draw.ellipse((src_px - rg, src_py - rg, src_px + rg, src_py + rg), outline=(190, 190, 190), width=2)
        draw.ellipse((src_px - 4, src_py - 4, src_px + 4, src_py + 4), fill=(0, 0, 0))

        if len(self._trail) > 1:
            pts = [self._world_to_px(tx, ty) for tx, ty in self._trail]
            draw.line(pts, fill=(80, 220, 255), width=2)

        ax, ay = self._world_to_px(self.x, self.y)
        size = 10
        p0 = (ax + size * np.cos(self.th), ay - size * np.sin(self.th))
        p1 = (ax + size * np.cos(self.th + 2.5), ay - size * np.sin(self.th + 2.5))
        p2 = (ax + size * np.cos(self.th - 2.5), ay - size * np.sin(self.th - 2.5))
        tri = [tuple(map(int, p0)), tuple(map(int, p1)), tuple(map(int, p2))]
        draw.polygon(tri, fill=(50, 100, 220))

        if self._sense_pt is not None:
            sx, sy = self._world_to_px(self._sense_pt[0], self._sense_pt[1])
            draw.line((ax, ay, sx, sy), fill=(220, 60, 60), width=2)
            draw.ellipse((sx - 3, sy - 3, sx + 3, sy + 3), fill=(220, 60, 60))

        return np.array(img, dtype=np.uint8)

    def close(self):
        self._heatmap_img = None
        self._cbar_img = None


In [ ]:
# Ported: connectome2 actor + recurrent critic.


def _build_gru(input_size, hidden_size):
    return nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)


class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=147):
        super().__init__()
        self.rnn = _build_gru(obs_dim, hidden)
        self.fc1 = nn.Linear(hidden + act_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.q = nn.Linear(hidden, 1)

    def forward(self, obs, act, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        if act.dim() == 2:
            act = act.unsqueeze(1)
        y, h2 = self.rnn(obs, h)
        z = torch.cat([y, act], dim=-1)
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        q = self.q(z).squeeze(-1)
        return q, h2


CONNECTOME_BASE = 90
CONNECTOME_RATIOS = (24, 7, 4, 54, 1)


def _population_sizes(total_size):
    if total_size % CONNECTOME_BASE != 0:
        raise ValueError(
            f"Connectome hidden_size must be a multiple of {CONNECTOME_BASE}; got {total_size}."
        )
    scale = total_size // CONNECTOME_BASE
    if scale < 2:
        raise ValueError(
            f"Connectome requires hidden_size >= {2 * CONNECTOME_BASE}; got {total_size}."
        )
    n_orn = CONNECTOME_RATIOS[0] * scale
    n_pn = CONNECTOME_RATIOS[1] * scale
    n_ln = CONNECTOME_RATIOS[2] * scale
    n_kc = CONNECTOME_RATIOS[3] * scale
    n_mbon = CONNECTOME_RATIOS[4] * scale
    return n_orn, n_pn, n_ln, n_kc, n_mbon


class _Backbone(nn.Module):
    def __init__(self, obs_dim, hidden_size, inner_steps=4):
        super().__init__()
        self.hidden_size = int(hidden_size)
        self.inner_steps = int(max(1, inner_steps))

        n_orn, n_pn, n_ln, n_kc, n_mbon = _population_sizes(self.hidden_size)
        self.n_orn = n_orn
        self.n_pn = n_pn
        self.n_ln = n_ln
        self.n_kc = n_kc
        self.n_mbon = n_mbon

        self.in_orn = nn.Linear(obs_dim, n_orn)
        self.W_oto = nn.Linear(n_orn, n_orn, bias=False)
        self.W_lto = nn.Linear(n_ln, n_orn, bias=False)

        self.W_otp = nn.Linear(n_orn, n_pn, bias=False)
        self.W_ltp = nn.Linear(n_ln, n_pn, bias=False)
        self.W_ptp = nn.Linear(n_pn, n_pn, bias=False)

        self.W_otl = nn.Linear(n_orn, n_ln, bias=False)
        self.W_ptl = nn.Linear(n_pn, n_ln, bias=False)
        self.W_ltl = nn.Linear(n_ln, n_ln, bias=False)

        self.W_ktk = nn.Linear(n_kc, n_kc, bias=False)
        self.W_mtk = nn.Linear(n_mbon, n_kc, bias=False)
        self.W_ptk = nn.Linear(n_pn, n_kc, bias=False)

        self.W_ktm = nn.Linear(n_kc, n_mbon, bias=False)

        self.b_orn = nn.Parameter(torch.zeros(n_orn))
        self.b_pn = nn.Parameter(torch.zeros(n_pn))
        self.b_ln = nn.Parameter(torch.zeros(n_ln))
        self.b_kc = nn.Parameter(torch.zeros(n_kc))
        self.b_mbon = nn.Parameter(torch.zeros(n_mbon))

        self.readout = nn.Linear(n_mbon, self.hidden_size)

    def _split_state(self, h_flat):
        i0 = self.n_orn
        i1 = i0 + self.n_pn
        i2 = i1 + self.n_ln
        i3 = i2 + self.n_kc
        h_orn = h_flat[:, :i0]
        h_pn = h_flat[:, i0:i1]
        h_ln = h_flat[:, i1:i2]
        h_kc = h_flat[:, i2:i3]
        h_mbon = h_flat[:, i3:]
        return h_orn, h_pn, h_ln, h_kc, h_mbon

    def _pack_state(self, h_orn, h_pn, h_ln, h_kc, h_mbon):
        return torch.cat([h_orn, h_pn, h_ln, h_kc, h_mbon], dim=-1)

    def _init_state(self, batch_size, device, dtype):
        return torch.zeros(batch_size, self.hidden_size, device=device, dtype=dtype)

    def _parse_hidden(self, h, batch_size, device, dtype):
        if h is None:
            return self._init_state(batch_size, device, dtype)
        if h.dim() == 3:
            if h.size(0) != 1:
                raise ValueError("Connectome hidden state expects first dim 1.")
            h_flat = h[0]
        elif h.dim() == 2:
            h_flat = h
        else:
            raise ValueError("Connectome hidden state must be rank-2 or rank-3 tensor.")
        if h_flat.size(0) != batch_size or h_flat.size(1) != self.hidden_size:
            raise ValueError(
                f"Connectome hidden state shape mismatch: got {tuple(h_flat.shape)}, expected ({batch_size}, {self.hidden_size})."
            )
        return h_flat.to(device=device, dtype=dtype)

    def step(self, h_orn, h_pn, h_ln, h_kc, h_mbon, x_t):
        orn_next = torch.tanh(self.W_oto(h_orn) + self.W_lto(h_ln) + x_t + self.b_orn)
        pn_next = torch.tanh(self.W_otp(orn_next) + self.W_ltp(h_ln) + self.W_ptp(h_pn) + self.b_pn)
        ln_next = torch.tanh(self.W_otl(orn_next) + self.W_ptl(pn_next) + self.W_ltl(h_ln) + self.b_ln)
        kc_next = torch.tanh(self.W_ktk(h_kc) + self.W_mtk(h_mbon) + self.W_ptk(pn_next) + self.b_kc)
        mbon_next = torch.tanh(self.W_ktm(kc_next) + self.b_mbon)
        return orn_next, pn_next, ln_next, kc_next, mbon_next

    def forward(self, obs, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        batch_size, seq_len, _ = obs.shape
        device = obs.device
        dtype = obs.dtype

        h_flat = self._parse_hidden(h, batch_size, device, dtype)
        h_orn, h_pn, h_ln, h_kc, h_mbon = self._split_state(h_flat)

        outputs = []
        for t in range(seq_len):
            x_t = self.in_orn(obs[:, t, :])
            for _ in range(self.inner_steps):
                h_orn, h_pn, h_ln, h_kc, h_mbon = self.step(h_orn, h_pn, h_ln, h_kc, h_mbon, x_t)
            outputs.append(self.readout(h_mbon).unsqueeze(1))

        y = torch.cat(outputs, dim=1)
        h2 = self._pack_state(h_orn, h_pn, h_ln, h_kc, h_mbon).unsqueeze(0)
        return y, h2


class Actor(nn.Module):
    def __init__(
        self,
        obs_dim,
        cont_act_dim,
        hidden=180,
        log_std_min=-5.0,
        log_std_max=2.0,
        cast_temperature=0.5,
        connectome_steps=4,
    ):
        super().__init__()
        self.backbone = _Backbone(obs_dim=obs_dim, hidden_size=hidden, inner_steps=connectome_steps)
        self.mu = nn.Linear(hidden, cont_act_dim)
        self.log_std = nn.Linear(hidden, cont_act_dim)
        self.cast_logit = nn.Linear(hidden, 1)
        self.log_std_min = float(log_std_min)
        self.log_std_max = float(log_std_max)
        self.cast_temperature = float(max(cast_temperature, 1e-3))

    def forward(self, obs, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        y, h2 = self.backbone(obs, h)
        mu = self.mu(y)
        log_std = torch.clamp(self.log_std(y), self.log_std_min, self.log_std_max)
        cast_logit = self.cast_logit(y)
        return mu, log_std, cast_logit, h2

    def sample(self, obs, action_low, action_high, h=None):
        mu, log_std, cast_logit, h2 = self.forward(obs, h)
        std = log_std.exp()
        normal = Normal(mu, std)
        x = normal.rsample()
        y = torch.tanh(x)

        cont_low = action_low[:2]
        cont_high = action_high[:2]
        cont_scale = (cont_high - cont_low) * 0.5
        cont_bias = (cont_high + cont_low) * 0.5
        cont_action = y * cont_scale + cont_bias

        cont_log_prob = normal.log_prob(x) - torch.log(cont_scale * (1.0 - y.pow(2)) + 1e-6)
        cont_log_prob = cont_log_prob.sum(dim=-1, keepdim=True)

        u = torch.rand_like(cast_logit)
        u = torch.clamp(u, 1e-6, 1.0 - 1e-6)
        logistic_noise = torch.log(u) - torch.log1p(-u)
        cast_soft = torch.sigmoid((cast_logit + logistic_noise) / self.cast_temperature)
        cast_hard = (cast_soft > 0.5).float()
        cast_action = cast_hard + cast_soft - cast_soft.detach()

        bern = Bernoulli(logits=cast_logit)
        disc_log_prob = bern.log_prob(cast_hard)

        action = torch.cat([cont_action, cast_action], dim=-1)
        log_prob = (cont_log_prob + disc_log_prob).squeeze(-1)
        cast_prob = torch.sigmoid(cast_logit)
        return action, log_prob, h2, mu, cast_prob

    def deterministic(self, obs, action_low, action_high, h=None):
        mu, _, cast_logit, h2 = self.forward(obs, h)
        y = torch.tanh(mu)

        cont_low = action_low[:2]
        cont_high = action_high[:2]
        cont_scale = (cont_high - cont_low) * 0.5
        cont_bias = (cont_high + cont_low) * 0.5
        cont_action = y * cont_scale + cont_bias

        cast_action = (cast_logit > 0.0).float()
        action = torch.cat([cont_action, cast_action], dim=-1)
        return action, h2


In [ ]:
# Ported: connectome2-only agent.


class Agent:
    def __init__(
        self,
        obs_dim,
        act_dim,
        action_low,
        action_high,
        device,
        critic_hidden=147,
        connectome_hidden=180,
        connectome_steps=4,
        lr_actor=3e-4,
        lr_critic=3e-4,
        lr_alpha=3e-4,
        gamma=0.99,
        tau=0.005,
    ):
        self.device = device
        self.gamma = float(gamma)
        self.tau = float(tau)
        self.act_dim = int(act_dim)
        if self.act_dim != 3:
            raise ValueError("Agent expects action dim exactly 3: [v, omega, cast].")

        self.action_low = torch.as_tensor(action_low, dtype=torch.float32, device=device)
        self.action_high = torch.as_tensor(action_high, dtype=torch.float32, device=device)
        if self.action_low.numel() != 3 or self.action_high.numel() != 3:
            raise ValueError("Agent expects action bounds with 3 elements: [v, omega, cast].")

        self.actor = Actor(
            obs_dim,
            cont_act_dim=2,
            hidden=connectome_hidden,
            connectome_steps=connectome_steps,
        ).to(device)
        self.q1 = Critic(obs_dim, act_dim, hidden=critic_hidden).to(device)
        self.q2 = Critic(obs_dim, act_dim, hidden=critic_hidden).to(device)
        self.tq1 = Critic(obs_dim, act_dim, hidden=critic_hidden).to(device)
        self.tq2 = Critic(obs_dim, act_dim, hidden=critic_hidden).to(device)
        self.tq1.load_state_dict(self.q1.state_dict())
        self.tq2.load_state_dict(self.q2.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_opt = optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()), lr=lr_critic)
        self.log_alpha = torch.zeros(1, device=device, requires_grad=True)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=lr_alpha)
        self.target_entropy = -(2.0 + 0.5 * float(np.log(2.0)))
        self.loss_fn = nn.SmoothL1Loss()

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def get_action(self, obs, h=None):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            action, _, h2, _, _ = self.actor.sample(obs_t, self.action_low, self.action_high, h)
        a = action[:, -1, :].squeeze(0).detach().cpu().numpy().astype(np.float32)
        return a, h2

    def get_action_deterministic(self, obs, h=None):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            action, h2 = self.actor.deterministic(obs_t, self.action_low, self.action_high, h)
        a = action[:, -1, :].squeeze(0).detach().cpu().numpy().astype(np.float32)
        return a, h2

    def update(self, batch):
        obs_seq, act_seq, rew_seq, terminal_seq = batch
        obs_seq = torch.as_tensor(obs_seq, dtype=torch.float32, device=self.device)
        act_seq = torch.as_tensor(act_seq, dtype=torch.float32, device=self.device)
        rew_seq = torch.as_tensor(rew_seq, dtype=torch.float32, device=self.device)
        terminal_seq = torch.as_tensor(terminal_seq, dtype=torch.float32, device=self.device)

        obs_t = obs_seq[:, :-1, :]
        next_obs_t = obs_seq[:, 1:, :]

        with torch.no_grad():
            next_a, next_logp, _, _, _ = self.actor.sample(next_obs_t, self.action_low, self.action_high, None)
            next_q1, _ = self.tq1(next_obs_t, next_a, None)
            next_q2, _ = self.tq2(next_obs_t, next_a, None)
            next_q = torch.min(next_q1, next_q2) - self.alpha.detach() * next_logp
            y = rew_seq + self.gamma * (1.0 - terminal_seq) * next_q

        q1, _ = self.q1(obs_t, act_seq, None)
        q2, _ = self.q2(obs_t, act_seq, None)
        critic_loss = self.loss_fn(q1, y) + self.loss_fn(q2, y)

        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        pi_a, logp, _, _, _ = self.actor.sample(obs_t, self.action_low, self.action_high, None)
        q1_pi, _ = self.q1(obs_t, pi_a, None)
        q2_pi, _ = self.q2(obs_t, pi_a, None)
        q_pi = torch.min(q1_pi, q2_pi)
        actor_loss = (self.alpha.detach() * logp - q_pi).mean()

        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        self.soft_update()
        td_err = (y - q1).abs().mean().item()
        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "alpha_loss": float(alpha_loss.item()),
            "alpha": float(self.alpha.item()),
            "td_abs": float(td_err),
        }

    def soft_update(self):
        with torch.no_grad():
            for p, tp in zip(self.q1.parameters(), self.tq1.parameters()):
                tp.data.mul_(1.0 - self.tau).add_(self.tau * p.data)
            for p, tp in zip(self.q2.parameters(), self.tq2.parameters()):
                tp.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

    def save(self, path):
        torch.save(
            {
                "actor": self.actor.state_dict(),
                "q1": self.q1.state_dict(),
                "q2": self.q2.state_dict(),
                "tq1": self.tq1.state_dict(),
                "tq2": self.tq2.state_dict(),
                "log_alpha": self.log_alpha.detach().cpu(),
            },
            path,
        )

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.actor.load_state_dict(ckpt["actor"])
        self.q1.load_state_dict(ckpt["q1"])
        self.q2.load_state_dict(ckpt["q2"])
        self.tq1.load_state_dict(ckpt.get("tq1", ckpt["q1"]))
        self.tq2.load_state_dict(ckpt.get("tq2", ckpt["q2"]))
        if "log_alpha" in ckpt:
            self.log_alpha.data.copy_(ckpt["log_alpha"].to(self.device))


# Notebook-local builders.
def make_env(env_config=None, render_mode=None):
    cfg = current_env_config() if env_config is None else dict(env_config)
    if render_mode is not None:
        cfg["render_mode"] = render_mode
    return Odor2DEnvConnectome(**cfg)


def build_agent(critic_hidden, connectome_hidden, connectome_steps, lr_actor, lr_critic, lr_alpha, env, device):
    return Agent(
        env.observation_space.shape[0],
        env.action_space.shape[0],
        env.action_space.low,
        env.action_space.high,
        device,
        critic_hidden=int(critic_hidden),
        connectome_hidden=int(connectome_hidden),
        connectome_steps=int(connectome_steps),
        lr_actor=float(lr_actor),
        lr_critic=float(lr_critic),
        lr_alpha=float(lr_alpha),
        gamma=GAMMA,
        tau=TAU,
    )


def make_agent(args, env, device):
    return build_agent(
        args.critic_hidden,
        args.connectome_hidden,
        args.connectome_steps,
        args.lr_actor,
        args.lr_critic,
        args.lr_alpha,
        env,
        device,
    )


def make_agent_from_run_config(run_config, env, device):
    return build_agent(
        run_config.get("critic_hidden", CRITIC_HIDDEN),
        run_config.get("connectome_hidden", CONNECTOME_HIDDEN),
        run_config.get("connectome_steps", CONNECTOME_STEPS),
        run_config.get("lr_actor", LR_ACTOR),
        run_config.get("lr_critic", LR_CRITIC),
        run_config.get("lr_alpha", LR_ALPHA),
        env,
        device,
    )


In [ ]:
# Inline plotting helpers.


def _ema(values, alpha=0.05):
    out = []
    m = float(values[0]) if values else 0.0
    for value in values:
        m = alpha * float(value) + (1.0 - alpha) * m
        out.append(float(m))
    return out


def show_training_plots(ep_returns, ep_steps_to_goal, ema_alpha=0.05):
    if not ep_returns:
        print("No training history to plot.")
        return

    xs_ret = np.arange(1, len(ep_returns) + 1)
    xs_step = np.arange(1, len(ep_steps_to_goal) + 1)
    ret_curve = _ema(ep_returns, ema_alpha)
    step_curve = _ema(ep_steps_to_goal, ema_alpha) if ep_steps_to_goal else []

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(xs_ret, ret_curve, linewidth=2)
    axes[0].set_title("Return")
    axes[0].set_xlabel("Episode")
    axes[0].grid(alpha=0.3)

    if ep_steps_to_goal:
        axes[1].plot(xs_step, step_curve, linewidth=2)
    axes[1].set_title("Step To Goal")
    axes[1].set_xlabel("Episode")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def training_history_path(run_dir):
    return Path(run_dir) / "train_history.json"


def save_training_history(run_dir, ep_returns, ep_steps_to_goal):
    payload = {
        "returns": [float(v) for v in ep_returns],
        "steps_to_goal": [int(v) for v in ep_steps_to_goal],
    }
    path = training_history_path(run_dir)
    path.write_text(json.dumps(payload))
    return payload


def load_training_history(run_dir):
    path = training_history_path(run_dir)
    if not path.exists():
        return {"returns": [], "steps_to_goal": []}
    data = json.loads(path.read_text())
    return {
        "returns": [float(v) for v in data.get("returns", [])],
        "steps_to_goal": [int(v) for v in data.get("steps_to_goal", [])],
    }


def show_training_plots_from_run_dir(run_dir, ema_alpha=0.05):
    history = load_training_history(run_dir)
    show_training_plots(history["returns"], history["steps_to_goal"], ema_alpha=ema_alpha)
    return history


def show_trajs(env_config, trajs, title):
    L = float(env_config["L"])
    src_x = float(env_config["src_x"])
    src_y = float(env_config["src_y"])

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_aspect("equal")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_color("white")

    res = 100
    xs = np.linspace(-L, L, res)
    ys = np.linspace(-L, L, res)
    X, Y = np.meshgrid(xs, ys)
    sigma_c = float(env_config.get("sigma_c", 1.0))
    bg_c = float(env_config.get("bg_c", 0.0))
    wind_x = float(env_config.get("wind_x", 0.0))
    wind_y = float(env_config.get("wind_y", 0.0))
    wind_mag = float(np.hypot(wind_x, wind_y))

    xr = X - src_x
    yr = Y - src_y
    if wind_mag <= 1e-6:
        r2 = xr * xr + yr * yr
        c = np.exp(-r2 / (2.0 * sigma_c * sigma_c))
    else:
        wx, wy = wind_x / wind_mag, wind_y / wind_mag
        t = xr * wx + yr * wy
        s = -xr * wy + yr * wx
        stretch = 1.0 + min(wind_mag, 2.0)
        sigma_s = sigma_c
        sigma_t = sigma_c * stretch
        sigma_up = sigma_c / stretch
        t_pos = np.maximum(0.0, t)
        t_neg = np.maximum(0.0, -t)
        c = np.exp(
            -(
                (s * s) / (2.0 * sigma_s**2)
                + (t_pos**2) / (2.0 * sigma_t**2)
                + (t_neg**2) / (2.0 * sigma_up**2)
            )
        )
    c = bg_c + (1.0 - bg_c) * c
    c = np.clip(c, 0.0, 1.0)
    im = ax.imshow(c, extent=[-L, L, -L, L], origin="lower", cmap="inferno", alpha=1.0, vmin=0.0, vmax=1.0)

    ax.plot(src_x, src_y, "ko")
    circle = plt.Circle((src_x, src_y), float(env_config["r_goal"]), color="gray", fill=False)
    ax.add_patch(circle)

    for traj in trajs:
        ax.plot(traj["x"], traj["y"], alpha=0.6)
        ax.plot(traj["x"][0], traj["y"][0], "x")
        ax.plot(traj["x"][-1], traj["y"][-1], "s")

    rets = [traj["return"] for traj in trajs]
    avg_ret = float(np.mean(rets)) if rets else float("nan")
    ax.set_title(f"{title}\nAvg Return: {avg_ret:.2f}")
    ax.title.set_color("white")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 1.0])
    cbar.set_ticklabels(["0.0", "1.0"])
    cbar.ax.tick_params(colors="white")
    cbar.outline.set_edgecolor("white")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def print_eval_summary(label, result):
    stats = result["stats"]
    print(f"[{label}] Avg Return: {result['avg_return']:.2f}")
    print(f"[{label}] Entry Success: {stats['success_entry_rate'] * 100:.1f}%")
    print(f"[{label}] Hold Success:  {stats['success_hold_rate'] * 100:.1f}%")
    print(f"[{label}] Final In-Goal: {stats['final_in_goal_rate'] * 100:.1f}%")
    print(f"[{label}] Avg Cast Starts: {stats['cast_start_count_mean']:.2f} / episode")
    print(f"[{label}] Avg Cast Steps:  {stats['cast_step_count_mean']:.2f} / episode")
    print(f"[{label}] Cast Step %:  {stats['cast_step_ratio_mean'] * 100:.1f}%")
    print(f"[{label}] Can-Turn %:   {stats['can_turn_ratio_mean'] * 100:.1f}%")


def show_eval_result_plots(eval_result):
    if not isinstance(eval_result, dict) or not eval_result.get("results"):
        print("No eval trajectories in memory yet.")
        return
    env_config = eval_result["env"]
    for label, result in eval_result["results"].items():
        show_trajs(env_config, result["trajectories"], f"Eval {label}")


def show_gif_frames(frames, fps=30):
    if not frames:
        print("No GIF frames to display.")
        return None

    buf = io.BytesIO()
    imageio.mimsave(buf, frames, format="GIF", fps=fps, loop=0)
    data = buf.getvalue()
    if DisplayImage is None or display is None:
        print("GIF prepared in memory.")
        return data
    display(DisplayImage(data=data))
    return data


def render_rollout_frame_png_style(env, title=None):
    L = float(env.L)
    src_x, src_y = float(env.src_x), float(env.src_y)

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_color("white")

    res = 100
    xs = np.linspace(-L, L, res)
    ys = np.linspace(-L, L, res)
    X, Y = np.meshgrid(xs, ys)
    c = np.asarray(env._conc(X, Y), dtype=np.float32)
    c = np.clip(c, 0.0, 1.0)
    im = ax.imshow(c, extent=[-L, L, -L, L], origin="lower", cmap="inferno", alpha=1.0, vmin=0.0, vmax=1.0)

    ax.plot(src_x, src_y, "ko")
    circle = plt.Circle((src_x, src_y), float(env.r_goal), color="gray", fill=False)
    ax.add_patch(circle)

    if len(env._trail) > 0:
        tx = [float(p[0]) for p in env._trail]
        ty = [float(p[1]) for p in env._trail]
        ax.plot(tx, ty, alpha=0.8, color="#50dcff")
        ax.plot(tx[0], ty[0], "x", color="white")
        ax.plot(tx[-1], ty[-1], "s", color="white", markersize=5)

    ax_x = float(env.x)
    ax_y = float(env.y)
    th = float(env.th)
    tri_len = 0.18
    p0 = (ax_x + tri_len * np.cos(th), ax_y + tri_len * np.sin(th))
    p1 = (ax_x + tri_len * np.cos(th + 2.5), ax_y + tri_len * np.sin(th + 2.5))
    p2 = (ax_x + tri_len * np.cos(th - 2.5), ax_y + tri_len * np.sin(th - 2.5))
    tri = plt.Polygon(
        [p0, p1, p2],
        closed=True,
        facecolor=(50 / 255.0, 100 / 255.0, 220 / 255.0),
        edgecolor="white",
        linewidth=0.8,
    )
    ax.add_patch(tri)

    if getattr(env, "_sense_pt", None) is not None:
        sx, sy = float(env._sense_pt[0]), float(env._sense_pt[1])
        ax.plot([ax_x, sx], [ax_y, sy], color=(220 / 255.0, 60 / 255.0, 60 / 255.0), linewidth=2)
        ax.plot(sx, sy, "o", color=(220 / 255.0, 60 / 255.0, 60 / 255.0), markersize=4)
        labels = ("F", "L", "R")
        scan_idx_attr = getattr(env, "_render_scan_idx", None)
        if scan_idx_attr is not None:
            scan_idx = max(0, min(int(scan_idx_attr), 2))
        else:
            sense_ang = float(np.arctan2(sy - ax_y, sx - ax_x))
            rel_ang = (sense_ang - th + np.pi) % (2.0 * np.pi) - np.pi
            if abs(rel_ang) < 1e-3:
                scan_idx = 0
            elif rel_ang > 0.0:
                scan_idx = 1
            else:
                scan_idx = 2
        ax.text(
            ax_x + 0.06 * L,
            ax_y + 0.06 * L,
            labels[scan_idx],
            color="white",
            fontsize=10,
            ha="left",
            va="bottom",
            bbox=dict(facecolor="black", edgecolor="none", alpha=0.4, pad=1.0),
        )

    if title:
        ax.set_title(title, color="white")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 1.0])
    cbar.set_ticklabels(["0.0", "1.0"])
    cbar.ax.tick_params(colors="white")
    cbar.outline.set_edgecolor("white")

    fig.tight_layout()
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frame = rgba[..., :3].copy()
    plt.close(fig)
    return frame


In [ ]:
# Train + eval.


def rollout(env, agent, episodes, seed_base):
    trajectories = []
    success_entry_count = 0
    success_hold_count = 0
    final_in_goal_count = 0
    cast_start_counts = []
    cast_step_counts = []
    cast_step_ratios = []
    can_turn_ratios = []
    base_env = env.unwrapped

    for i in range(episodes):
        ep_seed = int(seed_base + i)
        obs, _ = env.reset(seed=ep_seed)
        h = None
        done = False
        xs, ys = [base_env.x], [base_env.y]
        ep_ret = 0.0
        in_goal = False
        hold_success = False
        final_in_goal = False
        cast_start_count = 0
        cast_step_count = 0
        cast_steps = 0
        can_turn_steps = 0
        total_steps = 0

        while not done:
            action, h = agent.get_action_deterministic(obs, h)
            obs, r, term, trunc, info = env.step(action)
            done = bool(term or trunc)
            total_steps += 1
            cast_start_count += int(info.get("cast_start", 0))
            cast_step_count += int(info.get("did_cast", 0))
            cast_steps += int(info.get("in_cast", 0))
            can_turn_steps += int(info.get("can_turn", 0))
            xs.append(base_env.x)
            ys.append(base_env.y)
            ep_ret += r
            if info.get("in_goal", 0):
                in_goal = True
            if info.get("success_hold", 0):
                hold_success = True
            final_in_goal = bool(info.get("in_goal", 0))

        if in_goal:
            success_entry_count += 1
        if hold_success:
            success_hold_count += 1
        if final_in_goal:
            final_in_goal_count += 1
        cast_start_counts.append(float(cast_start_count))
        cast_step_counts.append(float(cast_step_count))
        if total_steps > 0:
            cast_step_ratios.append(float(cast_steps) / float(total_steps))
            can_turn_ratios.append(float(can_turn_steps) / float(total_steps))
        else:
            cast_step_ratios.append(0.0)
            can_turn_ratios.append(0.0)
        trajectories.append(
            {
                "return": ep_ret,
                "success": in_goal,
                "x": xs,
                "y": ys,
                "seed": ep_seed,
            }
        )

    stats = {
        "success_entry_rate": float(success_entry_count / episodes) if episodes > 0 else 0.0,
        "success_hold_rate": float(success_hold_count / episodes) if episodes > 0 else 0.0,
        "final_in_goal_rate": float(final_in_goal_count / episodes) if episodes > 0 else 0.0,
        "cast_start_count_mean": float(np.mean(cast_start_counts)) if cast_start_counts else 0.0,
        "cast_step_count_mean": float(np.mean(cast_step_counts)) if cast_step_counts else 0.0,
        "cast_step_ratio_mean": float(np.mean(cast_step_ratios)) if cast_step_ratios else 0.0,
        "can_turn_ratio_mean": float(np.mean(can_turn_ratios)) if can_turn_ratios else 0.0,
    }
    return trajectories, stats



def select_eval_checkpoints(run_dir):
    ckpt_dir = Path(run_dir) / "checkpoints"
    specs = []
    missing = []
    for label in ["first", "best"]:
        path = ckpt_dir / f"{label}.pt"
        if path.exists():
            specs.append((label, str(path)))
        else:
            missing.append(label)
    if missing:
        missing_text = ", ".join(f"{label}.pt" for label in missing)
        raise FileNotFoundError(f"Missing checkpoint(s) under {ckpt_dir}: {missing_text}")
    return specs



def train(args=None):
    global LAST_RUN_DIR

    if args is None:
        args = make_args()

    set_seed(args.seed)

    run_dir = run_dir_from_args(args)
    LAST_RUN_DIR = run_dir
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    write_run_config(run_dir, args)

    device = torch.device("cuda" if torch.cuda.is_available() and not FORCE_CPU else "cpu")
    env = make_env()
    env.action_space.seed(args.seed)
    if hasattr(env, "observation_space") and hasattr(env.observation_space, "seed"):
        env.observation_space.seed(args.seed)
    agent = make_agent(args, env, device)
    buffer = ReplayBuffer(BUFFER_SIZE)

    first_path = os.path.join(ckpt_dir, "first.pt")
    best_path = os.path.join(ckpt_dir, "best.pt")
    ep_returns = []
    ep_steps_to_goal = []
    best_return = -np.inf
    first_ep = None
    best_ep = None
    interrupted = False

    try:
        for ep in range(1, args.total_episodes + 1):
            obs, _ = env.reset(seed=args.seed + ep)
            h = None
            done = False
            ep_ret = 0.0
            traj = []
            step_count = 0
            first_goal_step = None
            log_stats = None

            while not done:
                action, h_next = agent.get_action(obs, h)
                next_obs, reward, term, trunc, info = env.step(action)
                done = bool(term or trunc)
                terminal = float(term)
                step_count += 1

                if first_goal_step is None and info.get("in_goal", 0):
                    first_goal_step = step_count

                traj.append((obs, action, float(reward), next_obs, terminal))
                obs = next_obs
                h = h_next
                ep_ret += reward

                if len(buffer) > args.learning_starts and len(buffer) > args.batch_size * args.seq_len:
                    batch = buffer.sample_continuous(args.batch_size, args.seq_len)
                    log_stats = agent.update(batch)

            buffer.add_episode(traj)
            ep_returns.append(float(ep_ret))
            ep_steps_to_goal.append(first_goal_step if first_goal_step is not None else step_count)

            if first_ep is None:
                agent.save(first_path)
                first_ep = ep

            if best_ep is None or ep_ret > best_return:
                best_return = ep_ret
                best_ep = ep
                agent.save(best_path)

            if ep % args.log_every == 0:
                avg_ret = float(np.mean(ep_returns[-args.log_every:]))
                avg_steps = float(np.mean(ep_steps_to_goal[-args.log_every:]))
                if isinstance(log_stats, dict):
                    print(
                        f"Ep {ep} | Avg Ret: {avg_ret:.2f} | Avg Step-to-Goal: {avg_steps:.1f} "
                        f"| alpha: {log_stats['alpha']:.4f} | td|: {log_stats['td_abs']:.4f}"
                    )
                else:
                    print(f"Ep {ep} | Avg Ret: {avg_ret:.2f} | Avg Step-to-Goal: {avg_steps:.1f}")
                save_training_history(run_dir, ep_returns, ep_steps_to_goal)
    except KeyboardInterrupt:
        interrupted = True
        print()
        print("[Warn] Training interrupted. Saved checkpoints can still be evaluated below.")
    finally:
        if first_ep is None:
            agent.save(first_path)
            first_ep = 0
        if best_ep is None:
            agent.save(best_path)
            best_ep = first_ep
        save_training_history(run_dir, ep_returns, ep_steps_to_goal)
        env.close()

    write_run_config(run_dir, args)
    return {
        "run_dir": run_dir,
        "interrupted": interrupted,
        "first_ep": int(first_ep),
        "best_ep": int(best_ep),
        "history_path": str(training_history_path(run_dir)),
    }



def eval(run_dir=None):
    global LAST_EVAL_RESULT

    run_dir = resolve_run_dir(run_dir)
    if not run_dir:
        raise ValueError("No run_dir available. Run train() first.")

    run_config = load_run_config(run_dir)
    history = show_training_plots_from_run_dir(run_dir)
    env_config = dict(run_config.get("env", current_env_config()))
    device = torch.device("cuda" if torch.cuda.is_available() and not FORCE_CPU else "cpu")
    print(f"[Info] Evaluating on {device}")

    seed = int(run_config.get("seed", SEED))
    set_seed(seed)
    env = make_env(env_config)
    env.action_space.seed(seed)
    if hasattr(env, "observation_space") and hasattr(env.observation_space, "seed"):
        env.observation_space.seed(seed)
    agent = make_agent_from_run_config(run_config, env, device)

    episodes = int(run_config.get("eval_episodes", EVAL_EPISODES))
    results = {}
    for label, ckpt_path in select_eval_checkpoints(run_dir):
        print(f"[Info] Loading model from {ckpt_path}")
        agent.load(ckpt_path)
        print(f"[Info] Starting evaluation for {label} over {episodes} episodes...")
        trajectories, rollout_stats = rollout(env, agent, episodes, EVAL_SEED_BASE)
        best_traj = max(trajectories, key=lambda traj: float(traj["return"])) if trajectories else None
        results[label] = {
            "checkpoint_path": ckpt_path,
            "trajectories": trajectories,
            "stats": rollout_stats,
            "avg_return": float(np.mean([traj["return"] for traj in trajectories])) if trajectories else 0.0,
            "best_seed": int(best_traj["seed"]) if best_traj is not None else None,
            "best_return": float(best_traj["return"]) if best_traj is not None else None,
        }
        print_eval_summary(label, results[label])

    env.close()

    LAST_EVAL_RESULT = {
        "run_dir": run_dir,
        "env": env_config,
        "config": run_config,
        "history": history,
        "results": results,
    }

    show_eval_result_plots(LAST_EVAL_RESULT)

    for label, result in LAST_EVAL_RESULT["results"].items():
        agent.load(result["checkpoint_path"])
        env_gif = make_env(env_config, render_mode="rgb_array")
        gif_seed = result.get("best_seed") if result.get("best_seed") is not None else int(EVAL_SEED_BASE)
        gif_return = result.get("best_return") if result.get("best_return") is not None else float("nan")
        frames = []
        obs, _ = env_gif.reset(seed=gif_seed)
        h = None
        done = False

        title = f"Eval GIF: {label} | best return seed={gif_seed} | return={gif_return:.2f}"
        frames.append(render_rollout_frame_png_style(env_gif, title=title))
        while not done:
            action, h = agent.get_action_deterministic(obs, h)
            obs, _, term, trunc, _ = env_gif.step(action)
            done = bool(term or trunc)
            frames.append(render_rollout_frame_png_style(env_gif, title=title))
        env_gif.close()

        print(f"[{label}] GIF")
        show_gif_frames(frames, fps=30)

    return LAST_EVAL_RESULT


In [ ]:
args = make_args()
run_dir = run_dir_from_args(args)
LAST_RUN_DIR = run_dir
set_seed(args.seed)

print("Training config:")
for key in [
    "total_episodes",
    "eval_episodes",
    "seed",
    "critic_hidden",
    "connectome_hidden",
    "connectome_steps",
    "lr_actor",
    "lr_critic",
    "lr_alpha",
    "batch_size",
    "seq_len",
    "learning_starts",
    "log_every",
    "run_name",
]:
    print(f"  {key}: {getattr(args, key)}")
print("  env:", current_env_config())
print("  run_dir:", run_dir)

train_result = train(args)
run_dir = train_result.get("run_dir") if isinstance(train_result, dict) else run_dir
LAST_RUN_DIR = run_dir
print()
print("Training finished. run_dir:", run_dir)
if isinstance(train_result, dict) and train_result.get("interrupted"):
    print("Saved checkpoints are ready for the eval cell below.")


In [ ]:
print("Evaluation started...")
eval_result = eval()
print("Evaluation finished.")
print("Evaluated checkpoints:", ", ".join(eval_result["results"].keys()))


In [ ]:
run_dir = resolve_run_dir()
if not run_dir:
    print("No run_dir available yet. Run train() first.")
else:
    run_path = Path(run_dir)
    print()
    print("Run dir:", run_path)

    print()
    print("Saved files:")
    for name in ["config.json", "train_history.json"]:
        p = run_path / name
        print(" -", name if p.exists() else f"{name} (missing)")

    ckpt_dir = run_path / "checkpoints"
    print()
    print("Checkpoints:")
    for name in ["first.pt", "best.pt"]:
        p = ckpt_dir / name
        print(" -", name if p.exists() else f"{name} (missing)")
